# RAG Assignment: Baseline (Bi-Encoder) vs. Re-Ranking (Cross-Encoder)

**Course:** CL3410 Language Models and Agents  |  **Deadline:** 19 Oct 2026

Both pipelines run over the *exact same* chunked corpus:

* **Pipeline 1 (baseline):** bi-encoder embeddings -> cosine top-10 -> take top-3 -> LLM -> *Answer 1*
* **Pipeline 2 (re-rank):** same top-10 -> cross-encoder scores every (query, chunk) pair -> re-sort -> top-3 -> LLM (same prompt) -> *Answer 2*

**Document:** ICD-10-CM Official Guidelines for Coding and Reporting FY2026. **LLM:** local Qwen2.5-7B-Instruct, 4-bit, via MLX on the Apple GPU (free, no API key).

Sections: 0 Setup, 1 Document and chunking, 2 Index, 3 Pipelines, 4 LLM, 5 Failure queries (+ 5a SCAN), 6 Run and rank shifts, 7 Latency, 8 Scaling demo, 9 Export.

## 0. Setup

In [1]:
# Run once (Colab / local). Comment out if already installed.
# !pip install -q sentence-transformers pypdf faiss-cpu pandas numpy google-genai

In [2]:
import os, re, time, json, textwrap
import numpy as np
import pandas as pd
from pypdf import PdfReader

# ---------------- CONFIG: edit this cell only ----------------
PDF_PATH    = "icd_10_cm_october_2025_guidelines_0.pdf"   # ICD-10-CM Official Guidelines FY2026 (121 pages)
PAGE_START  = 7     # skip cover + table of contents (pages 1-6); Section I starts on page 7
PAGE_END    = None  # None = until the last page
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"   # bi-encoder
CE_MODEL    = "cross-encoder/ms-marco-MiniLM-L-6-v2"     # cross-encoder
TOP_K       = 10    # candidates retrieved by the bi-encoder (assignment: 10)
TOP_N       = 3     # chunks given to the LLM (assignment: 3)

CHUNK_WORDS   = 120   # target chunk size in words
CHUNK_OVERLAP = 25    # words shared between neighbouring chunks

LLM_BACKEND = "mlx"               # "mlx" (local model on Apple GPU, no API key) | "hf" | "gemini" | "openai" | "ollama"
LLM_MODEL   = "mlx-community/Qwen2.5-7B-Instruct-4bit"  # local 7B open-weights model; for "hf" e.g. "Qwen/Qwen2.5-1.5B-Instruct"
# API keys are read from the environment: GEMINI_API_KEY / OPENAI_API_KEY
# -------------------------------------------------------------
pd.set_option("display.max_colwidth", 120, "display.width", 200)

# ---- API key: read from env, else Colab secrets, else ask once (hidden input; never saved in the notebook) ----
import getpass
_KEY_VAR = {"gemini": "GEMINI_API_KEY", "openai": "OPENAI_API_KEY"}.get(LLM_BACKEND)
if _KEY_VAR and not os.environ.get(_KEY_VAR):
    try:
        from google.colab import userdata
        os.environ[_KEY_VAR] = userdata.get(_KEY_VAR)
    except Exception:
        os.environ[_KEY_VAR] = getpass.getpass(f"Paste your {_KEY_VAR} (get a free Gemini key at aistudio.google.com/apikey): ")

## 1. Document loading and chunking

The chunker works page by page (so every chunk has a page number we can cite), splits into sentences, and packs sentences into windows of about `CHUNK_WORDS` words with a small overlap. A chunk never crosses a page boundary. Both pipelines use this identical list `chunks`.

In [3]:
HEADER_RE = re.compile(r"ICD-10-CM Official Guidelines for Coding and Reporting\s*FY 20\d\d\s*Page \d+ of \d+")

def load_pages(path, start=PAGE_START, end=PAGE_END):
    reader = PdfReader(path)
    pages = []
    for i, p in enumerate(reader.pages, start=1):
        if i < start or (end and i > end):
            continue
        t = p.extract_text() or ""
        t = HEADER_RE.sub(" ", t)                # drop the running page header (same text on every page = noise)
        t = re.sub(r"-\n(?=[a-z])", "", t)      # re-join words hyphenated across lines
        t = re.sub(r"[ \t]+", " ", t)
        t = t.replace("\n", " ")
        t = re.sub(r"\s+", " ", t).strip()
        pages.append((i, t))
    return pages

def split_sentences(text):
    text = text.replace(" <PARA> ", " ")
    return [s.strip() for s in re.split(r"(?<=[.!?:;])\s+(?=[A-Z0-9\"'(])", text) if s.strip()]

def chunk_page(text, page_no, size=CHUNK_WORDS, overlap=CHUNK_OVERLAP):
    sents, out, cur, cur_w = split_sentences(text), [], [], 0
    for s in sents:
        w = len(s.split())
        if cur and cur_w + w > size:
            out.append(" ".join(cur))
            # carry the tail of the previous chunk forward as overlap
            tail, tw = [], 0
            for prev in reversed(cur):
                tw += len(prev.split())
                tail.insert(0, prev)
                if tw >= overlap:
                    break
            cur, cur_w = tail, sum(len(x.split()) for x in tail)
        cur.append(s); cur_w += w
    if cur:
        out.append(" ".join(cur))
    return [{"page": page_no, "text": c} for c in out if len(c.split()) >= 15]

pages = load_pages(PDF_PATH)
chunks = []
for pno, ptxt in pages:
    chunks += chunk_page(ptxt, pno)
for i, c in enumerate(chunks):
    c["id"] = i

lens = [len(c["text"].split()) for c in chunks]
print(f"pages={len(pages)}  chunks={len(chunks)}  words/chunk: min={min(lens)} mean={np.mean(lens):.0f} max={max(lens)}")
print("\nExample chunk:\n", textwrap.fill(chunks[len(chunks)//2]["text"], 110))

pages=115  chunks=601  words/chunk: min=26 mean=103 max=162

Example chunk:
 Assign 7th character “0”: • For single gestations • When the documentation in the record is insufficient to
determine the fetus affected and it is not possible to obtain clarification. • When it is not possible to
clinically determine which fetus is affected. 7) Completed weeks of gestation In ICD-10-CM, “completed” weeks
of gestation refers to full weeks. For example, if the provider documents gestation at 39 weeks and 6 days,
the code for 39 weeks of gestation should be assigned, as the patient has not yet reached 40 completed weeks.
b.


## 2. Index the chunks (bi-encoder + vector store)

Embeddings are L2-normalised, so inner product = cosine similarity. The vector store is an exact inner-product search over the embedding matrix (FAISS `IndexFlatIP` if installed, otherwise NumPy, which is what ran here). Exact search is fine at 601 chunks.

In [4]:
from sentence_transformers import SentenceTransformer, CrossEncoder
try:
    import faiss
except ImportError:
    faiss = None

bi_encoder = SentenceTransformer(EMBED_MODEL)
cross_encoder = CrossEncoder(CE_MODEL)

t0 = time.perf_counter()
emb = bi_encoder.encode([c["text"] for c in chunks], batch_size=32, normalize_embeddings=True,
                        show_progress_bar=False).astype("float32")
index_time = time.perf_counter() - t0
print(f"Indexed {len(chunks)} chunks, dim={emb.shape[1]}, one-off indexing time = {index_time:.2f}s")

if faiss is not None:
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(emb)
    def vector_search(qv, k):
        scores, ids = index.search(qv.reshape(1, -1), k)
        return ids[0].tolist(), scores[0].tolist()
else:  # numpy fallback, same maths
    def vector_search(qv, k):
        s = emb @ qv
        ids = np.argsort(-s)[:k]
        return ids.tolist(), s[ids].tolist()

  warnings.warn(


Indexed 601 chunks, dim=384, one-off indexing time = 2.12s


In [5]:
def vector_search(qv, k):
    s = np.einsum("ij,j->i", emb, qv)      # same maths as the index; einsum avoids a macOS Accelerate matmul bug that returned NaN scores
    ids = np.argsort(-s)[:k]
    return ids.tolist(), s[ids].tolist()

# sanity checks
q = bi_encoder.encode("test", normalize_embeddings=True).astype("float32")
print("embeddings finite:", np.isfinite(emb).all())
print("scores finite:", np.isfinite(np.einsum("ij,j->i", emb, q)).all())
print("chunk 0 vs itself (should be ~1.0):", vector_search(emb[0], 1))

embeddings finite: True
scores finite: True
chunk 0 vs itself (should be ~1.0): ([0], [1.0000001192092896])


## 3. The two retrieval pipelines

`baseline_retrieve` returns the Top-10 (this is the shared candidate list). Pipeline 1 uses its first 3. Pipeline 2 re-scores all 10 with the cross-encoder and keeps the best 3. Each function times itself.

In [6]:
def baseline_retrieve(query, k=TOP_K):
    """Stage 1: bi-encoder dense retrieval. Returns (list of (chunk_id, cosine)), seconds."""
    t0 = time.perf_counter()
    qv = bi_encoder.encode(query, normalize_embeddings=True).astype("float32")
    ids, scores = vector_search(qv, k)
    dt = time.perf_counter() - t0
    return list(zip(ids, scores)), dt

def rerank(query, candidates, n=TOP_N):
    """Stage 2: cross-encoder scores each (query, chunk) pair jointly, then re-sorts."""
    t0 = time.perf_counter()
    pairs = [(query, chunks[cid]["text"]) for cid, _ in candidates]
    ce_scores = cross_encoder.predict(pairs, show_progress_bar=False)
    order = np.argsort(-ce_scores)
    reranked = [(candidates[i][0], float(ce_scores[i])) for i in order]
    dt = time.perf_counter() - t0
    return reranked, dt

## 4. LLM answer generation (identical prompt for both pipelines)

The prompt builder is one function, used unchanged for Answer 1 and Answer 2. The only thing that differs is which 3 chunks go in. Decoding is greedy (temperature 0) and answers are cached so reruns are consistent. The LLM is identical in both pipelines, so any difference between Answer 1 and Answer 2 comes from the retrieved chunks.

In [7]:
SYSTEM_PROMPT = (
    "You answer questions about a document using ONLY the numbered passages provided. "
    "Cite passages like [1]. If the passages do not contain the answer, say exactly: "
    "\"The provided passages do not answer this.\" Do not use outside knowledge."
)

def build_prompt(query, chunk_ids):
    ctx = "\n\n".join(f"[{i+1}] (page {chunks[cid]['page']}) {chunks[cid]['text']}"
                       for i, cid in enumerate(chunk_ids))
    return f"Passages:\n{ctx}\n\nQuestion: {query}\nAnswer:"

_cache_path = "llm_cache.json"
_cache = json.load(open(_cache_path)) if os.path.exists(_cache_path) else {}

def call_llm(prompt):
    key = f"{LLM_BACKEND}|{LLM_MODEL}|{prompt}"
    if key in _cache:
        return _cache[key]
    if LLM_BACKEND == "gemini":
        from google import genai
        from google.genai import types
        client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
        for attempt in range(6):          # free tier is rate-limited: back off and retry on 429 / 503
            try:
                r = client.models.generate_content(
                    model=LLM_MODEL, contents=prompt,
                    config=types.GenerateContentConfig(system_instruction=SYSTEM_PROMPT, temperature=0))
                break
            except Exception as e:
                if attempt == 5 or not any(t in str(e) for t in ("429", "503", "RESOURCE_EXHAUSTED", "UNAVAILABLE")):
                    raise
                time.sleep(15 * (attempt + 1))
        out = r.text.strip()
    elif LLM_BACKEND == "openai":
        from openai import OpenAI
        r = OpenAI().chat.completions.create(
            model=LLM_MODEL, temperature=0,
            messages=[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": prompt}])
        out = r.choices[0].message.content.strip()
    elif LLM_BACKEND == "mlx":     # local open-weights model on the Apple GPU via mlx-lm (free, no key); greedy decoding
        global _mlx
        if "_mlx" not in globals():
            from mlx_lm import load
            _mlx = load(LLM_MODEL)
        from mlx_lm import generate
        lm, tok = _mlx
        text = tok.apply_chat_template([{"role": "system", "content": SYSTEM_PROMPT},
                                        {"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)
        out = generate(lm, tok, prompt=text, max_tokens=250).strip()
    elif LLM_BACKEND == "hf":      # local open-weights model through transformers (free, no key); greedy decoding
        global _hf
        if "_hf" not in globals():
            import torch
            from transformers import AutoTokenizer, AutoModelForCausalLM
            _hf = (AutoTokenizer.from_pretrained(LLM_MODEL),
                   AutoModelForCausalLM.from_pretrained(LLM_MODEL, torch_dtype=torch.float32))
        tok, lm = _hf
        text = tok.apply_chat_template([{"role": "system", "content": SYSTEM_PROMPT},
                                        {"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)
        inp = tok(text, return_tensors="pt")
        gen = lm.generate(**inp, max_new_tokens=250, do_sample=False)
        out = tok.decode(gen[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    elif LLM_BACKEND == "ollama":
        import requests
        r = requests.post("http://localhost:11434/api/chat", json={
            "model": LLM_MODEL, "stream": False, "options": {"temperature": 0},
            "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": prompt}]})
        out = r.json()["message"]["content"].strip()
    else:
        raise ValueError(LLM_BACKEND)
    _cache[key] = out
    json.dump(_cache, open(_cache_path, "w"))
    return out

## 5. Failure-mode queries

Explore first with `compare(query)`, which prints the baseline top-10 next to the cross-encoder order. Once a query is found where the right chunk is buried (baseline rank 4-10) and gets promoted, add it to `TEST_QUERIES`.

Each test query needs:
* `category`: Keyword Trap, Exception/Corner Case, Negation/Constraint, or Other
* `query`
* `gold`: a regex that matches text found **only** in the chunk(s) that truly answer the question (this lets the notebook compute ranks automatically; check it by eye with `compare`)
* `expected`: the correct answer in a line, written by you from the document (used when judging Answer 1 vs Answer 2)

In [8]:
def compare(query, k=TOP_K, show_chars=160):
    """Interactive helper: baseline order vs cross-encoder order for any query."""
    cands, _ = baseline_retrieve(query, k)
    rr, _ = rerank(query, cands)
    base_rank = {cid: r for r, (cid, _) in enumerate(cands, 1)}
    print(f"QUERY: {query}\n")
    print("Cross-encoder order (new rank | old rank | CE score | cosine | chunk id, page):")
    cos = dict(cands)
    for r, (cid, s) in enumerate(rr, 1):
        txt = chunks[cid]["text"][:show_chars].replace("\n", " ")
        print(f"  {r:>2} | was {base_rank[cid]:>2} | CE {s:7.2f} | cos {cos[cid]:.3f} | #{cid} p{chunks[cid]['page']}: {txt}...")

# compare("your exploratory query here")

### 5a. SCAN: searching for queries that re-ranking rescues

My first 9 hand-written queries produced no rescue, so this cell scans a larger batch of paraphrased queries. Every rescue was then checked by eye with `compare()` before being added to `TEST_QUERIES`. One flagged query (influenza / pneumonia) was dropped because its gold chunk was about VAP, not influenza.

The same procedure (scan, then check by eye with `compare()`) was repeated on further batches of negation-style and keyword-trap phrasings; the rescues it found are in `TEST_QUERIES`. It also surfaced cases where the cross-encoder **falls for the same keyword overlap** and pushes the right chunk down; two are kept as failures.


In [9]:
# SCAN: run many candidate queries through both retrievers (no LLM calls) and flag "rescues":
# gold chunk at baseline rank 4-10 that the cross-encoder moves into the top 3.
# Queries are paraphrased on purpose (synonyms, negations, exceptions) so the bi-encoder has to work harder.
SCAN_CANDIDATES = [  # (category, query, gold regex)
    ('Keyword Trap', 'Is a borderline diagnosis at discharge coded as confirmed?', r'borderline. diagnosis at the time of discharge'),
    ('Keyword Trap', "The doctor says 'possible Zika'. Do I use the Zika code?", r'probable. Zika, do not assign'),
    ('Negation/Constraint', 'Can an injury suspected to be caused by terrorism get a terrorism code?', r'suspected to be the result of terrorism'),
    ('Negation/Constraint', 'Is a dam collapse code used if a hurricane caused the dam to fail?', r'Collapse of dam'),
    ('Negation/Constraint', 'Can an underdosing code be listed first?', r'underdosing should never be assigned as principal'),
    ('Negation/Constraint', 'Are codes for diseases classified elsewhere allowed as the principal diagnosis?', r'classified elsewhere. codes are never permitted'),
    ('Negation/Constraint', "Can obstetric chapter codes appear on a baby's birth record?", r'never permitted on the newborn record'),
    ('Negation/Constraint', 'A patient had a stroke but has no lasting neurological deficits. Should a late effects of stroke code be used?', r'I69 should not be assigned if the patient does not have neurologic deficits'),
    ('Negation/Constraint', 'Should normal pain right after an operation be coded?', r'Routine or expected postoperative pain immediately after surgery should not be coded'),
    ('Negation/Constraint', 'Should abnormal lab results be coded if the provider has not commented on them?', r'Abnormal findings .laboratory'),
    ('Negation/Constraint', 'A kidney transplant patient still has chronic kidney disease. Is this a transplant complication?', r'T86\\.1- should not be assigned for post kidney transplant'),
    ('Negation/Constraint', 'Should prematurity be coded for a newborn if the doctor did not document it?', r'prematurity should not be assigned unless it is documented'),
    ('Exception/Corner Case', 'Is the unspecified activity code used when the activity was not recorded?', r'Do not assign Y93\\.9'),
    ('Exception/Corner Case', 'Should Glasgow coma scale scores be reported for a sedated patient?', r'medically induced coma or a sedated patient'),
    ('Exception/Corner Case', 'Are bruises and scrapes coded when there is a more serious injury at the same place?', r'Superficial injuries such as abrasions or contusions are not coded'),
    ('Exception/Corner Case', 'Is the aftercare Z code used for a healing traumatic fracture?', r'aftercare Z codes should not be used for aftercare for traumatic fractures'),
    ('Exception/Corner Case', 'Do burn codes cover sunburn?', r'thermal burns, except sunburns'),
    ('Exception/Corner Case', 'Can the unspecified multiple injuries category T07 be used for inpatients?', r'T07, Unspecified multiple injuries should not be assigned in the inpatient'),
    ('Exception/Corner Case', 'Can T31 or T32 extent-of-burn codes be used for burn sequelae?', r'T31 and T32 should not be used for sequelae'),
    ('Exception/Corner Case', 'Is a prophylactic organ removal code used when testes are removed to treat prostate cancer?', r'Z40\\.0 code should not be assigned if the patient is having organ removal for treatment of a malignancy'),
    ('Exception/Corner Case', 'For a suspected child abuse case, should the perpetrator be coded?', r'suspected cases of abuse or neglect, do not report external cause or perpetrator'),
    ('Exception/Corner Case', 'Should conditions treated in the past that are now gone be coded?', r'Do not code conditions that were previously treated and no longer exist'),
    ('Exception/Corner Case', 'Is the Y99 external cause status code required if there is no other external cause code?', r'Do not assign a code from category Y99 if no other external cause'),
    ('Exception/Corner Case', 'Should a perinatal condition code still be used when the patient is an adult?', r'perinatal code should continue to be used regardless of the patient'),
    ('Exception/Corner Case', 'If two drugs were taken together in a poisoning, are they coded separately?', r'code each individually unless a combination code'),
    ('Exception/Corner Case', 'Does sepsis after an obstetric procedure use code O85?', r'O85 should not be assigned for sepsis following an obstetrical procedure'),
    ('Exception/Corner Case', 'Should a status code be added when the diagnosis code already contains that information?', r'status code should not be used with a diagnosis code from one of the body system'),
    ('Exception/Corner Case', 'Is a ventilator associated pneumonia code assigned without the provider stating VAP?', r'J95\\.851, Ventilator associated pneumonia, should be assigned only when'),
    ('Exception/Corner Case', 'Should I add a separate code for the type of pneumonia with influenza with pneumonia?', r'Do not assign an additional code from categories J12-J18'),
    ('Exception/Corner Case', 'Are features that are inherent to a chromosomal abnormality coded separately?', r'manifestations that are an inherent component of the anomaly should not be coded separately'),
    ('Keyword Trap', 'Is hypertensive CKD coded if the doctor says the kidney disease is unrelated to the high blood pressure?', r'CKD should not be coded as hypertensive if the provider indicates'),
    ('Keyword Trap', 'Is a link between hypertension and heart disease presumed without documentation?', r'coded as related even in the absence of provider documentation'),
    ('Keyword Trap', 'Can a patient already diagnosed with HIV illness be coded as asymptomatic HIV later?', r'previously diagnosed with any HIV illness .B20. should never be assigned'),
    ('Keyword Trap', 'Which external cause takes priority over transport accidents?', r'transport accidents take priority over all other external cause codes except'),
    ('Keyword Trap', 'Do signs routinely associated with a disease get extra codes?', r'Signs or symptoms that are associated routinely with a disease process should not be assigned'),
    ('Other', 'How many place of occurrence codes can one encounter have?', r'only one place of occurrence code'),
    ('Other', 'Is type 2 MI after a type 1 MI coded with I22?', r'Do not assign code I22 for subsequent myocardial infarctions other than type 1'),
    ('Other', 'How is demand ischemia coded?', r'Do not assign code I24\\.89'),
    ('Other', 'Can chemotherapy admission be principal when the neoplasm is being treated?', r'only exception to this guideline is if the administration of chemotherapy'),
    ('Other', 'Is reporting of external cause codes mandatory?', r'reporting of ICD-10-CM codes in Chapter 20, External Causes of Morbidity, is not required'),
    ('Other', 'What 7th characters exist for injury categories?', r'three 7th character values .with the exception of fractures'),
    ('Other', 'Is code Z79 used for a short course of medication?', r'Do not assign a code from category Z79 for medication being administered for a brief'),
    ('Other', 'Can a three-character code be used if it has subcategories?', r'three-character code is to be used only if it is not further subdivided'),
    ('Other', 'When an HIV patient is admitted for hemolytic-uremic syndrome which is sequenced first?', r'hemolytic-uremic syndrome associated with HIV'),
]

def _rank(ids, rx):
    return next((r for r, cid in enumerate(ids, 1) if re.search(rx, chunks[cid]["text"], re.I | re.S)), None)

scan_rows = []
for cat, q, rx in SCAN_CANDIDATES:
    cands, _ = baseline_retrieve(q)
    b = _rank([c for c, _ in cands], rx)
    r = _rank([c for c, _ in rerank(q, cands)[0]], rx)
    scan_rows.append({"category": cat, "query": q, "baseline_rank": b or "miss", "rerank_rank": r or "miss",
                      "rescued": bool(b and b > TOP_N and r and r <= TOP_N)})
scan = pd.DataFrame(scan_rows)
display(scan)
print(f"Rescues: {scan.rescued.sum()} / {len(scan)}")


,category,query,baseline_rank,rerank_rank,rescued
0,Keyword Trap,Is a borderline diagnosis at discharge coded as confirmed?,1,1,False
1,Keyword Trap,The doctor says 'possible Zika'. Do I use the Zika code?,1,1,False
2,Negation/Constraint,Can an injury suspected to be caused by terrorism get a terrorism code?,1,1,False
3,Negation/Constraint,Is a dam collapse code used if a hurricane caused the dam to fail?,1,1,False
4,Negation/Constraint,Can an underdosing code be listed first?,1,1,False
5,Negation/Constraint,Are codes for diseases classified elsewhere allowed as the principal diagnosis?,1,1,False
6,Negation/Constraint,Can obstetric chapter codes appear on a baby's birth record?,1,1,False
7,Negation/Constraint,A patient had a stroke but has no lasting neurological deficits. Should a late effects of stroke code be used?,miss,miss,False
8,Negation/Constraint,Should normal pain right after an operation be coded?,1,1,False
9,Negation/Constraint,Should abnormal lab results be coded if the provider has not commented on them?,1,1,False


Rescues: 6 / 44


In [10]:
# ---- Test queries. The first 8 were found with the SCAN cell above: the gold chunk sits at baseline rank 4-10
# ---- and the cross-encoder promotes it into the top 3 (the 'rescues' the assignment asks for).
# ---- The rest are kept because they show other behaviour: easy wins, partial moves, misses, and one demotion.
TEST_QUERIES = [
    # ---- Rescued by re-ranking ----
    {"category": 'Exception/Corner Case',
     "query": 'Can T31 or T32 extent-of-burn codes be used for burn sequelae?',
     "gold": r'T31 and T32 should not be used for sequelae',
     "expected": 'No.'},
    {"category": 'Exception/Corner Case',
     "query": 'Should a perinatal condition code still be used when the patient is an adult?',
     "gold": r'perinatal code should continue to be used regardless of the patient',
     "expected": 'Yes; a perinatal-origin condition keeps the Chapter 16 code at any age.'},
    {"category": 'Exception/Corner Case',
     "query": 'If two drugs were taken together in a poisoning, are they coded separately?',
     "gold": r'code each individually unless a combination code',
     "expected": 'Yes, each individually unless a combination code exists in the Table of Drugs.'},
    {"category": 'Other (distractor chunks)',
     "query": 'Can a patient already diagnosed with HIV illness be coded as asymptomatic HIV later?',
     "gold": r'previously diagnosed with any HIV illness .B20. should never be assigned',
     "expected": 'No; never R75 or Z21 once diagnosed with HIV illness.'},
    {"category": 'Other (distractor chunks)',
     "query": 'Do signs routinely associated with a disease get extra codes?',
     "gold": r'Signs or symptoms that are associated routinely with a disease process should not be assigned',
     "expected": 'No, unless instructed by the classification.'},
    {"category": 'Negation/Constraint',
     "query": 'Which codes are not allowed to be reported when the cause of a collapse is a hurricane?',
     "gold": r'X36\.0.-, Collapse of dam or manmade structure, should not be assigned',
     "expected": 'X36.0.- (collapse of dam/man-made structure) is not assigned when the collapse is due to the hurricane; it is limited to collapses from earth surface movements.'},
    {"category": 'Negation/Constraint',
     "query": 'Should obstetric complication codes not be used for sepsis after delivery procedures?',
     "gold": r'O85 should not be assigned for sepsis following an obstetrical procedure',
     "expected": 'Correct: O85 is not assigned for sepsis following an obstetrical procedure; use the postprocedural sepsis guidance (e.g. O86.04).'},
    {"category": 'Keyword Trap',
     "query": 'How long is 7th character A used for a fracture caused by disease such as osteoporosis?',
     "gold": r'7th character A is for use as long as the patient is receiving active treatment for the fracture',
     "expected": 'Pathologic fracture: 7th character A is used as long as the patient is receiving active treatment for the fracture.'},
    # ---- Re-ranking makes it worse: the cross-encoder also falls for the keyword overlap ----
    {"category": 'Keyword Trap (re-ranker trapped)',
     "query": 'A woman wants to give one of her kidneys to her brother. How is her visit coded?',
     "gold": r'Z52, Donors of organs and tissues',
     "expected": 'A code from category Z52, Donors of organs and tissues (living donor).'},
    {"category": 'Keyword Trap (re-ranker trapped)',
     "query": 'Patient had a heart attack two years ago and needs no more treatment for it. Which heart attack code is used?',
     "gold": r'I25\.2, Old myocardial infarction',
     "expected": 'I25.2, Old myocardial infarction (healed, no further care needed).'},
    # ---- Other failure types / controls ----
    {"category": "Keyword Trap",
     "query": "In an outpatient visit, how should a suspected or probable diagnosis be reported?",
     "gold": r"Do not code diagnoses documented as .probable",
     "expected": "Outpatient: do NOT code it; code signs/symptoms or the reason for the visit instead (inpatient is the opposite)."},
    {"category": "Keyword Trap",
     "query": "Can a probable diagnosis at discharge be coded as if it exists for an inpatient admission?",
     "gold": r"code the condition as if it existed or was established",
     "expected": "Yes: for inpatient short-term acute/LTC/psychiatric hospitals, code uncertain diagnoses as if established."},
    {"category": "Exception/Corner Case",
     "query": "Can two codes with an Excludes1 note ever be reported together?",
     "gold": r"exception to the Excludes1 definition",
     "expected": "Yes, if the two conditions are unrelated (e.g. F45.8 and G47.63 for psychogenic dysmenorrhea plus teeth grinding)."},
    {"category": "Negation/Constraint",
     "query": "Can severe sepsis be the principal diagnosis?",
     "gold": r"R65\.2 can never be assigned as a principal",
     "expected": "No: R65.2 can never be principal; the underlying systemic infection is sequenced first."},
    {"category": "Negation/Constraint",
     "query": "Should Z79.4 long-term insulin use be assigned when insulin is only given temporarily during the encounter?",
     "gold": r"Z79\.4 should not be assigned if insulin is given\s+temporarily",
     "expected": "No: not assigned if insulin is given temporarily to bring blood sugar under control during an encounter."},
    {"category": "Negation/Constraint",
     "query": "The doctor documents possible COVID-19. Can U07.1 be assigned?",
     "gold": r"do not assign code U07\.1",
     "expected": "No: only confirmed COVID-19; code the signs/symptoms (and exposure code if applicable)."},
    {"category": "Other (buried rule)",
     "query": "A patient has MRSA pneumonia. Should an extra MRSA infection code be added?",
     "gold": r"Do not assign code B95\.62",
     "expected": "No: use the combination code J15.212; do not add B95.62 or Z16.11."},
    {"category": "Other (buried rule)",
     "query": "Provider documents both alcohol use and alcohol dependence. Which code is assigned?",
     "gold": r"If both use and dependence are documented",
     "expected": "Only the dependence code, per the use/abuse/dependence hierarchy."},
    {"category": "Other (buried rule)",
     "query": "Which clinical information can be taken from clinicians who are not the patient's provider?",
     "gold": r"Depth of non-pressure chronic ulcers",
     "expected": "BMI, pressure ulcer stage, depth of chronic ulcers, coma scale, NIHSS, SDOH, laterality, blood alcohol, underimmunization, firearm injury intent."},
]
assert TEST_QUERIES, "Add your test queries above (see the comment template)."

## 6. Run both pipelines and measure rank shifts

For every test query this records: baseline rank of the gold chunk (1-10, or "miss" if outside top-10), its rank after re-ranking, whether it is in the top-3 for each pipeline, and both LLM answers.

In [11]:
def gold_rank(ranked_ids, gold_regex):
    """1-based position of the first chunk in `ranked_ids` whose text matches gold_regex; None if absent."""
    for r, cid in enumerate(ranked_ids, 1):
        if re.search(gold_regex, chunks[cid]["text"], re.I | re.S):
            return r
    return None

rows, details = [], {}
for q in TEST_QUERIES:
    cands, t_base = baseline_retrieve(q["query"])
    base_ids = [cid for cid, _ in cands]
    rr, t_rr = rerank(q["query"], cands)
    rr_ids = [cid for cid, _ in rr]

    ids1, ids2 = base_ids[:TOP_N], rr_ids[:TOP_N]
    ans1 = call_llm(build_prompt(q["query"], ids1))
    ans2 = call_llm(build_prompt(q["query"], ids2))   # identical prompt structure

    b, r = gold_rank(base_ids, q["gold"]), gold_rank(rr_ids, q["gold"])
    rows.append({
        "category": q["category"], "query": q["query"],
        "baseline_rank": b if b else "miss", "rerank_rank": r if r else "miss",
        "gold_in_top3_baseline": bool(b and b <= TOP_N), "gold_in_top3_rerank": bool(r and r <= TOP_N),
        "rescued": bool(b and b > TOP_N and r and r <= TOP_N),
    })
    details[q["query"]] = dict(ids1=ids1, ids2=ids2, ans1=ans1, ans2=ans2, expected=q["expected"])

results = pd.DataFrame(rows)
display(results)
print(f"Rescued by re-ranking (gold moved from rank >{TOP_N} into top-{TOP_N}): {results['rescued'].sum()} / {len(results)}")

,category,query,baseline_rank,rerank_rank,gold_in_top3_baseline,gold_in_top3_rerank,rescued
0,Exception/Corner Case,Can T31 or T32 extent-of-burn codes be used for burn sequelae?,9,1,False,True,True
1,Exception/Corner Case,Should a perinatal condition code still be used when the patient is an adult?,5,1,False,True,True
2,Exception/Corner Case,"If two drugs were taken together in a poisoning, are they coded separately?",4,1,False,True,True
3,Other (distractor chunks),Can a patient already diagnosed with HIV illness be coded as asymptomatic HIV later?,4,1,False,True,True
4,Other (distractor chunks),Do signs routinely associated with a disease get extra codes?,6,1,False,True,True
5,Negation/Constraint,Which codes are not allowed to be reported when the cause of a collapse is a hurricane?,4,1,False,True,True
6,Negation/Constraint,Should obstetric complication codes not be used for sepsis after delivery procedures?,8,1,False,True,True
7,Keyword Trap,How long is 7th character A used for a fracture caused by disease such as osteoporosis?,6,2,False,True,True
8,Keyword Trap (re-ranker trapped),A woman wants to give one of her kidneys to her brother. How is her visit coded?,1,7,True,False,False
9,Keyword Trap (re-ranker trapped),Patient had a heart attack two years ago and needs no more treatment for it. Which heart attack code is used?,2,6,True,False,False


Rescued by re-ranking (gold moved from rank >3 into top-3): 8 / 19


In [12]:
# Side-by-side answers for the report (read these and judge each yourself against `expected`)
for q, d in details.items():
    print("="*110)
    print("Q:", q)
    print("EXPECTED :", d["expected"])
    print(f"\nANSWER 1 (baseline, chunks {d['ids1']}):\n{textwrap.fill(d['ans1'], 110)}")
    print(f"\nANSWER 2 (re-ranked, chunks {d['ids2']}):\n{textwrap.fill(d['ans2'], 110)}")

Q: Can T31 or T32 extent-of-burn codes be used for burn sequelae?
EXPECTED : No.

ANSWER 1 (baseline, chunks [388, 390, 392]):
The provided passages do not answer this.

ANSWER 2 (re-ranked, chunks [389, 388, 387]):
The provided passages do not answer this.
Q: Should a perinatal condition code still be used when the patient is an adult?
EXPECTED : Yes; a perinatal-origin condition keeps the Chapter 16 code at any age.

ANSWER 1 (baseline, chunks [339, 349, 479]):
The provided passages do not answer this.

ANSWER 2 (re-ranked, chunks [341, 342, 339]):
Yes, a perinatal condition code should still be used when the patient is an adult if the condition originated
during the perinatal period and continues throughout the patient's life. [1]
Q: If two drugs were taken together in a poisoning, are they coded separately?
EXPECTED : Yes, each individually unless a combination code exists in the Table of Drugs.

ANSWER 1 (baseline, chunks [397, 398, 395]):
If two drugs were taken together in a poi

### 6a. Answer 1 vs Answer 2, judged by hand

Each answer was read against the `expected` line and the text of the gold chunk, and marked Correct, Partial, Wrong or No answer ("The provided passages do not answer this."). The LLM and prompt are identical for both pipelines, so a difference comes from the 3 chunks it was given.


In [13]:
judg = pd.read_csv("results/answer_judgements.csv")
display(judg[["category", "query", "baseline_rank", "rerank_rank", "answer1", "answer2", "note"]])
print("Answer 1:", judg.answer1.value_counts().to_dict())
print("Answer 2:", judg.answer2.value_counts().to_dict())
resc = judg.merge(results[["query", "rescued"]], on="query")
r = resc[resc.rescued]
print(f"On the {len(r)} rescued queries: Answer 1 correct {(r.answer1 == 'Correct').sum()}, Answer 2 correct {(r.answer2 == 'Correct').sum()}")


,category,query,baseline_rank,rerank_rank,answer1,answer2,note
0,Exception/Corner Case,Can T31 or T32 extent-of-burn codes be used for burn sequelae?,9,1,No answer,No answer,LLM refused even though the gold chunk was ranked 1 for Answer 2
1,Exception/Corner Case,Should a perinatal condition code still be used when the patient is an adult?,5,1,No answer,Correct,Gold chunk outside Answer 1's top 3
2,Exception/Corner Case,"If two drugs were taken together in a poisoning, are they coded separately?",4,1,Wrong,Wrong,A1 cites the multiple-unspecified-drugs rule; A2 misreads 'assign the code only once' despite having the gold chunk
3,Other (distractor chunks),Can a patient already diagnosed with HIV illness be coded as asymptomatic HIV later?,4,1,Wrong,Correct,A1 quotes the Z21 rule from a distractor chunk
4,Other (distractor chunks),Do signs routinely associated with a disease get extra codes?,6,1,Correct,Correct,A1's rule is not in its own passages (likely model prior knowledge)
5,Negation/Constraint,Which codes are not allowed to be reported when the cause of a collapse is a hurricane?,4,1,No answer,Correct,Gold chunk outside Answer 1's top 3
6,Negation/Constraint,Should obstetric complication codes not be used for sepsis after delivery procedures?,8,1,Wrong,Correct,A1 misstates the rule (says O85 should be assigned)
7,Keyword Trap,How long is 7th character A used for a fracture caused by disease such as osteoporosis?,6,2,No answer,Correct,Baseline top 3 dominated by traumatic-fracture chunks
8,Keyword Trap (re-ranker trapped),A woman wants to give one of her kidneys to her brother. How is her visit coded?,1,7,Correct,No answer,Re-ranking pushed the gold chunk from rank 1 to 7
9,Keyword Trap (re-ranker trapped),Patient had a heart attack two years ago and needs no more treatment for it. Which heart attack code is used?,2,6,Correct,No answer,Re-ranking pushed the gold chunk from rank 2 to 6


Answer 1: {'Correct': 9, 'Wrong': 5, 'No answer': 4, 'Partial': 1}
Answer 2: {'Correct': 12, 'No answer': 4, 'Wrong': 2, 'Partial': 1}
On the 8 rescued queries: Answer 1 correct 1, Answer 2 correct 6


## 7. Latency profiling

Each stage is timed over several repeats (first call is a warm-up and is dropped). Baseline = query embedding + vector search. Re-ranking = the extra cross-encoder pass over the 10 candidates. Pipeline 2 total = baseline + re-ranking. LLM time is excluded (it is the same for both and depends on the API). Both encoders run on the Apple M4 GPU (PyTorch MPS).


In [14]:
N_REPEATS = 10
base_t, rr_t = [], []
queries = [q["query"] for q in TEST_QUERIES]
_ = rerank(queries[0], baseline_retrieve(queries[0])[0])      # warm-up
for q in queries:
    for _ in range(N_REPEATS):
        cands, tb = baseline_retrieve(q)
        _, tr = rerank(q, cands)
        base_t.append(tb); rr_t.append(tr)

base_t, rr_t = np.array(base_t) * 1000, np.array(rr_t) * 1000      # ms
latency = pd.DataFrame({
    "Pipeline": ["1: Baseline (embed + search)", "2: Baseline + cross-encoder re-rank", "Re-rank step only"],
    "Mean (ms)":   [base_t.mean(), (base_t + rr_t).mean(), rr_t.mean()],
    "Std (ms)":    [base_t.std(),  (base_t + rr_t).std(),  rr_t.std()],
    "p95 (ms)":    [np.percentile(base_t, 95), np.percentile(base_t + rr_t, 95), np.percentile(rr_t, 95)],
}).round(2)
display(latency)
print(f"Re-ranking adds ~{rr_t.mean()/base_t.mean():.1f}x the baseline retrieval time "
      f"({len(chunks)} chunks, {N_REPEATS * len(queries)} timed runs). Device: {bi_encoder.device} ({os.cpu_count()} CPU cores).")

,Pipeline,Mean (ms),Std (ms),p95 (ms)
0,1: Baseline (embed + search),5.34,0.41,5.84
1,2: Baseline + cross-encoder re-rank,41.81,5.41,52.61
2,Re-rank step only,36.47,5.35,47.30


Re-ranking adds ~6.8x the baseline retrieval time (601 chunks, 190 timed runs). Device: mps:0 (10 CPU cores).


## 8. Scaling demo: why not cross-encode the whole collection?

Measure the cross-encoder on N pairs for increasing N, then extrapolate to corpus sizes. This gives real numbers for the scaling question.

In [15]:
q = queries[0]
sizes = [s for s in (10, 50, 100, 250, 500) if s <= len(chunks)]
if len(chunks) not in sizes:
    sizes.append(len(chunks))
cross_encoder.predict([(q, chunks[0]["text"])] * 8, show_progress_bar=False)   # warm-up, not timed
meas = []
for n in sizes:
    pairs = [(q, chunks[i]["text"]) for i in range(n)]
    t0 = time.perf_counter(); cross_encoder.predict(pairs, show_progress_bar=False)
    meas.append((n, time.perf_counter() - t0))
scal = pd.DataFrame(meas, columns=["pairs scored", "seconds"])
scal["ms per pair"] = (scal["seconds"] / scal["pairs scored"] * 1000).round(2)
display(scal)

per_pair = scal["seconds"].sum() / scal["pairs scored"].sum()
proj = pd.DataFrame({"corpus size (chunks)": [10_000, 100_000, 1_000_000, 10_000_000]})
proj["cross-encoder over everything, per query"] = [f"{per_pair * n / 60:.1f} min" if per_pair * n < 3600 else f"{per_pair * n / 3600:.1f} h"
                                                    for n in proj["corpus size (chunks)"]]
display(proj)
print(f"Measured ~{per_pair*1000:.1f} ms per (query, chunk) pair on this machine. "
      f"A bi-encoder instead compares the query with pre-computed vectors (one encode + one matrix search).")

,pairs scored,seconds,ms per pair
0,10,0.050045,5.00
1,50,0.181107,3.62
2,100,0.373254,3.73
3,250,0.963891,3.86
4,500,1.949460,3.90
5,601,2.537109,4.22


,corpus size (chunks),"cross-encoder over everything, per query"
0,10000,0.7 min
1,100000,6.7 min
2,1000000,1.1 h
3,10000000,11.1 h


Measured ~4.0 ms per (query, chunk) pair on this machine. A bi-encoder instead compares the query with pre-computed vectors (one encode + one matrix search).


## 9. Export results (tables for the report)

In [16]:
os.makedirs("results", exist_ok=True)
results.to_csv("results/rank_shifts.csv", index=False)
latency.to_csv("results/latency.csv", index=False)
scal.to_csv("results/scaling.csv", index=False)
with open("results/answers.md", "w") as f:
    for q, d in details.items():
        f.write(f"## {q}\n**Expected:** {d['expected']}\n\n**Answer 1:** {d['ans1']}\n\n**Answer 2:** {d['ans2']}\n\n")
print("Saved to results/: rank_shifts.csv, latency.csv, scaling.csv, answers.md")

Saved to results/: rank_shifts.csv, latency.csv, scaling.csv, answers.md
